In [1]:
import pandas as pd

In [2]:
# All prices in EUR: power & fuels in €/MWh, carbon in €/tCO2 (UK_CO2 already includes the £18 CPS)
data = pd.read_csv("../2_Transform/transformed_data.csv", index_col="date", parse_dates=True)

In [3]:
# Emission factors (tCO2 per MWh of fuel burned)
EF_GAS = 0.202
EF_COAL = 0.341
EF_OIL = 0.267

# Plant efficiencies (MWh of power out per MWh of fuel in)
EFF_GAS = 0.51          # implied efficiency calculated for another project
EFF_COAL = 0.35         # market convention
EFF_OIL = 0.35          # market convention

In [4]:
def clean_spread(power, fuel, co2, eff, ef):
    return power - (fuel + ef * co2) / eff


eu_spark_df = pd.DataFrame({
    "EU_Clean_Spark_Spread":  clean_spread(data["DE_POWER"], data["TTF_GAS"], data["EUA"], EFF_GAS,  EF_GAS),
    "EU_Clean_Dark_Spread":   clean_spread(data["DE_POWER"], data["COAL"],    data["EUA"], EFF_COAL, EF_COAL),
    "EU_Clean_Gasoil_Spread": clean_spread(data["DE_POWER"], data["GASOIL"],  data["EUA"], EFF_OIL,  EF_OIL),
})

uk_spark_df = pd.DataFrame({
    "UK_Clean_Spark_Spread":  clean_spread(data["UK_POWER"], data["NBP_GAS"], data["UK_CO2"], EFF_GAS,  EF_GAS),
    "UK_Clean_Dark_Spread":   clean_spread(data["UK_POWER"], data["COAL"],    data["UK_CO2"], EFF_COAL, EF_COAL),
    "UK_Clean_Gasoil_Spread": clean_spread(data["UK_POWER"], data["GASOIL"],  data["UK_CO2"], EFF_OIL,  EF_OIL),
})

In [5]:
eu_spark_df

,EU_Clean_Spark_Spread,EU_Clean_Dark_Spread,EU_Clean_Gasoil_Spread
date,,,
2019-01-02,1.719569,0.452094,-73.199426
2019-01-03,2.639294,2.837429,-70.492868
2019-01-04,0.595529,1.959696,-74.809749
2019-01-07,2.762980,3.124724,-77.181196
2019-01-08,0.819294,1.659175,-79.254603
...,...,...,...
2026-11-03,17.098902,93.426052,-142.436978
2026-11-04,25.259765,95.404707,-125.986309
2026-11-05,23.336863,99.598096,-122.875018


In [6]:
uk_spark_df

,UK_Clean_Spark_Spread,UK_Clean_Dark_Spread,UK_Clean_Gasoil_Spread
date,,,
2019-01-02,6.716440,-3.958714,-73.380159
2019-01-03,7.071946,-2.066179,-71.177936
2019-01-04,6.895306,-0.840410,-73.359608
2019-01-07,6.912223,-2.379535,-78.447089
2019-01-08,7.076425,-1.809974,-78.493958
...,...,...,...
2026-11-03,-9.874519,61.252977,-172.069184
2026-11-04,-5.121088,60.767850,-158.116555
2026-11-05,-3.101125,68.654299,-151.223753


In [7]:
# Save the two databases used by the models
eu_spark_df.to_csv("eu_spreads.csv")
uk_spark_df.to_csv("uk_spreads.csv")